In [35]:
import pandas as pd
import multiprocessing as mp
import time

In [36]:
# Load the dataset and confirm the number of rows and columns. Display the first 5 rows.

df = pd.read_csv(r"diabetes_dataset.csv")

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

df.head()

Total rows: 100000
Total columns: 16


,year,gender,age,location,race:AfricanAmerican,race:Asian,race:Caucasian,race:Hispanic,race:Other,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,0,0,1,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,1,0,0,0,0,0,never,19.95,5.0,90,0
2,2015,Male,18.0,Alabama,0,0,0,0,1,0,0,never,23.76,4.8,160,0
3,2015,Male,41.0,Alabama,0,0,1,0,0,0,0,never,27.32,4.0,159,0
4,2016,Female,52.0,Alabama,1,0,0,0,0,0,0,never,23.75,6.5,90,0


In [37]:
# Get only the necessary columns

df = df[['year', 'gender', 'age', 'location', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hbA1c_level', 'blood_glucose_level', 'diabetes']]
df.head()

,year,gender,age,location,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,0,never,19.95,5.0,90,0
2,2015,Male,18.0,Alabama,0,0,never,23.76,4.8,160,0
3,2015,Male,41.0,Alabama,0,0,never,27.32,4.0,159,0
4,2016,Female,52.0,Alabama,0,0,never,23.75,6.5,90,0


In [38]:
def sequential_compute(data):

    # Filters to get only the rows where the person has diabetes
    diabetes_data = data[data['diabetes'] == 1]

    # Aggregates the blood glucose levels and gets the average or the mean
    mean_glucose = diabetes_data['blood_glucose_level'].mean()

    # Sorts the diatebes dataset by BMI from highest to lowest
    sorted_data = diabetes_data.sort_values(by='bmi', ascending=False)

    return mean_glucose, sorted_data.head(10)

In [39]:
start = time.time()

seq_avg_glucose, seq_top10_bmi = sequential_compute(df)

end = time.time()

seq_time = end - start

print("Sequential Execution Time:", seq_time)
print("Sequential Average Glucose Level:", seq_avg_glucose)
print("Sequential Top 10 BMI Records:\n")
seq_top10_bmi

Sequential Execution Time: 0.007911443710327148
Sequential Average Glucose Level: 194.09470588235294
Sequential Top 10 BMI Records:



,year,gender,age,location,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
16294,2015,Female,45.0,District of Columbia,0,0,never,88.72,7.0,300,1
5766,2019,Male,49.0,Arizona,0,0,former,83.74,6.8,155,1
69209,2019,Female,48.0,North Carolina,0,0,never,81.73,6.5,130,1
46883,2019,Female,36.0,Michigan,0,0,never,79.46,6.2,220,1
22502,2019,Female,42.0,Guam,0,0,never,72.89,6.8,280,1
74977,2019,Female,52.0,Oklahoma,0,0,never,72.21,6.6,220,1
58119,2015,Male,38.0,Nevada,0,0,current,69.66,8.8,300,1
72259,2015,Female,46.0,Ohio,0,0,never,69.39,5.7,159,1
20273,2016,Male,43.0,Georgia,0,0,never,69.37,7.5,130,1
16165,2019,Female,54.0,Delaware,0,0,never,69.32,5.7,220,1


In [40]:
# Function that processes a chunk of the dataset

def chunk_data(chunk):

    diabetes_data = chunk[chunk['diabetes'] == 1]

    mean_glucose = diabetes_data['blood_glucose_level'].mean()
    count = len(diabetes_data)

    top10 = diabetes_data.nlargest(10, 'bmi')

    return mean_glucose, count, top10

In [41]:
def parallel_compute(data):

    diabetes_data = data[data['diabetes'] == 1]

    num_processes = min(8, mp.cpu_count())

    chunks = [diabetes_data.iloc[i::num_processes] for i in range(num_processes)]

    with mp.Pool(num_processes) as pool:
        results = pool.map(chunk_data, chunks)

    means, counts, top10_chunks = zip(*results)

    # Compute the mean
    total_sum = sum(m * c for m, c in zip(means, counts))
    total_count = sum(counts)
    par_mean = total_sum / total_count

    par_top10 = pd.concat(top10_chunks).nlargest(10, 'bmi')

    return par_mean, par_top10

In [43]:
start = time.time()

par_avg_glucose, par_top10_bmi = parallel_compute(df)

end = time.time()
parallel_time = end - start

print("Parallel Execution Time:", parallel_time)
print("Parallel Average Glucose Level:", par_avg_glucose)
print("Parallel Top 10 BMI Records:")
par_top10_bmi

Parallel Execution Time: 0.06970500946044922
Parallel Average Glucose Level: 194.09470588235294
Parallel Top 10 BMI Records:


,year,gender,age,location,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
16294,2015,Female,45.0,District of Columbia,0,0,never,88.72,7.0,300,1
5766,2019,Male,49.0,Arizona,0,0,former,83.74,6.8,155,1
69209,2019,Female,48.0,North Carolina,0,0,never,81.73,6.5,130,1
46883,2019,Female,36.0,Michigan,0,0,never,79.46,6.2,220,1
22502,2019,Female,42.0,Guam,0,0,never,72.89,6.8,280,1
74977,2019,Female,52.0,Oklahoma,0,0,never,72.21,6.6,220,1
58119,2015,Male,38.0,Nevada,0,0,current,69.66,8.8,300,1
72259,2015,Female,46.0,Ohio,0,0,never,69.39,5.7,159,1
20273,2016,Male,43.0,Georgia,0,0,never,69.37,7.5,130,1
16165,2019,Female,54.0,Delaware,0,0,never,69.32,5.7,220,1


In [44]:
speedup = seq_time / parallel_time
print("Speedup:", speedup)

Speedup: 0.1134989259963607


In [46]:
num_processors = mp.cpu_count()

efficiency = speedup / num_processors

print("Number of Processors:", num_processors)
print("Efficiency:", efficiency)

Number of Processors: 2
Efficiency: 0.05674946299818035


In [47]:
import pandas as pd

results = {
    "Version": ["Sequential", "Parallel"],
    "Execution Time (seconds)": [seq_time, parallel_time]
}

results_table = pd.DataFrame(results)
results_table

,Version,Execution Time (seconds)
0,Sequential,0.007911
1,Parallel,0.069705
